# Romanian Idiom Embeddings – Dissertation Notebook

# 1. Introduction

This notebook presents an experimental study on Romanian idiomatic expressions and figurative language using distributed semantic models.

The main goal is to analyze the relationship between:
- idiomatic expressions,
- their component words,
- their figurative meanings,
- and their contextual usage.

# 2. Dataset Description

The dataset contains Romanian idiomatic expressions manually annotated with:
- figurative meanings,
- opacity scores,
- expression type,
- and contextual sentences.

The dataset consists of over 300 expressions, making it suitable for statistical analysis.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

file_path = "romanian_idioms_dataset.xlsx"
df = pd.read_excel(file_path)

# Drop missing values BEFORE converting to string.
df = df.dropna(subset=["expression", "figurative_meaning", "opacity"])

# Basic cleaning
df["expression"] = df["expression"].astype(str).str.strip()
df["figurative_meaning"] = df["figurative_meaning"].astype(str).str.strip()
df["type"] = df["type"].astype(str).str.strip().str.lower()
df["context_sentence"] = df["context_sentence"].astype(str).str.strip()
df["opacity"] = pd.to_numeric(df["opacity"], errors="coerce")

# Remove possible invalid rows after numeric conversion
df = df.dropna(subset=["expression", "figurative_meaning", "opacity"])

print(df.shape)
df.head()

# 3. Exploratory Data Analysis

In this section, we explore:
- the distribution of manually annotated opacity scores;
- the distribution of expression types;
- the approximate length of idiomatic expressions.


In [ ]:
# Opacity distribution
df["opacity"].hist()
plt.title("Opacity Distribution")
plt.xlabel("Manual opacity score")
plt.ylabel("Number of expressions")
plt.show()

# Expression type distribution
print(df["type"].value_counts())

# Expression length, using simple whitespace tokenization
# This is kept simple to remain close to the original version.
df["expression_length"] = df["expression"].apply(lambda x: len(x.split()))
print(df["expression_length"].describe())

# 4. Models and Embeddings

We use the following models:
- SentenceTransformer (multilingual MiniLM)
- XLM-R (contextual transformer)
- Romanian BERT model

These models provide vector representations of expressions and words.

We use SentenceTransformer as the main embedding model.
The selected model, paraphrase-multilingual-MiniLM-L12-v2, is suitable for multilingual sentence-level semantic similarity.

This model is appropriate for comparing:
- the full idiomatic expression;
- the average embedding of its component words;
- the paraphrased figurative meaning;
- the context sentence.

In [ ]:
!pip install -q sentence-transformers scikit-learn openpyxl scipy

from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from scipy.stats import pearsonr, spearmanr, kruskal
from scipy.spatial.distance import cosine

st_model = SentenceTransformer("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")


# 5. Experiment 1: Compositionality

This experiment evaluates how well the meaning of an idiomatic expression can be approximated by combining the embeddings of its component words.

We compare:
- the embedding of the full expression;
- the mean embedding of its component words.

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

def tokenize_expression(expr):
    return expr.split()

def encode_text_st(text):
    return st_model.encode(text, convert_to_numpy=True)

def encode_words_st(words):
    return st_model.encode(words, convert_to_numpy=True)

def cosine_sim(vec1, vec2):
    return cosine_similarity([vec1], [vec2])[0][0]

# Reset lists before running the experiment
sim_expr_vs_mean = []

# Compute similarity between full expression and aggregated word embeddings
for expr in df["expression"]:
    words = tokenize_expression(expr)

    # Encode full expression
    expr_emb = encode_text_st(expr)

    # Encode individual component words
    word_embs = encode_words_st(words)

    # Aggregate word embeddings using mean
    mean_emb = np.mean(word_embs, axis=0)

    # Store similarity score
    sim_expr_vs_mean.append(cosine_sim(expr_emb, mean_emb))

print("len(df) =", len(df))
print("len(sim_expr_vs_mean) =", len(sim_expr_vs_mean))

# Save results to dataframe
df["st_sim_expr_vs_mean"] = sim_expr_vs_mean

print(df[[
    "expression",
    "st_sim_expr_vs_mean"
]].head())

The similarity between idiomatic expressions and the mean embedding of their component words is relatively high (typically between 0.57 and 0.82).
This suggests that the SentenceTransformer model tends to represent idiomatic expressions in a way that is still strongly influenced by their lexical components.
In other words, the model does not fully separate figurative meaning from literal composition, indicating partial compositional behavior.

# 6. Experiment 2: Automatic Opacity Estimation

Automatic opacity is computed as:

    opacity = 1 - cosine_similarity(expression, components)

Lower similarity indicates higher semantic opacity.

In [ ]:
# Automatic opacity
df["auto_opacity_mean"] = 1 - df["st_sim_expr_vs_mean"]

# Normalization between 0 and 1
def minmax_norm(series):
    if series.max() == series.min():
        return series * 0
    return (series - series.min()) / (series.max() - series.min())


df["auto_opacity_mean_norm"] = minmax_norm(df["auto_opacity_mean"])

print(df[["expression", "opacity", "auto_opacity_mean"]].head())


Comparison between manually annotated opacity and automatically estimated opacity.

Pearson correlation measures linear association.
Spearman correlation measures monotonic association and is especially useful here because opacity is an ordinal score.

In [ ]:

pearson_mean = pearsonr(df["opacity"], df["auto_opacity_mean"])
spearman_mean = spearmanr(df["opacity"], df["auto_opacity_mean"])

print("=== Manual opacity vs automatic opacity ===")
print("Pearson:", pearson_mean)
print("Spearman:", spearman_mean)

# Visualization
plt.figure(figsize=(8, 5))
plt.scatter(df["opacity"], df["auto_opacity_mean"], alpha=0.7)
plt.xlabel("Manual opacity")
plt.ylabel("Automatic opacity: 1 - sim(expression, mean components)")
plt.title("Manual opacity vs automatic opacity")
plt.grid(True)
plt.show()

The correlation between manually annotated opacity and automatically computed opacity is extremely weak (Pearson r ≈ 0.05, Spearman ρ ≈ 0.05) and not statistically significant.
This indicates that the embedding-based measure of opacity, defined as the inverse similarity between the expression and its components, does not reliably reflect human judgments of idiomatic opacity.

# 7. Experiment 3 - Expression vs Figurative Meaning

This experiment measures how similar the embedding of an idiomatic expression is to the embedding of its annotated figurative meaning.

For example, an expression such as "a da cu bâta-n baltă" should ideally be semantically close to a paraphrase such as "to make a serious mistake".

A random baseline is also computed by shuffling the figurative meanings. If the model captures meaningful semantic relationships, correct expression-meaning pairs should have higher similarity than random pairs.
"""


In [ ]:
# Embeddings: expression vs figurative meaning
expr_embs = st_model.encode(df["expression"].tolist(), convert_to_numpy=True)
meaning_embs = st_model.encode(df["figurative_meaning"].tolist(), convert_to_numpy=True)

# Cosine similarity for correct pairs
st_scores = [
    1 - cosine(expr_embs[i], meaning_embs[i]) for i in range(len(df))
]

df["st_expr_vs_meaning"] = st_scores

# Random baseline
shuffled_meanings = df["figurative_meaning"].sample(frac=1, random_state=42).tolist()
random_meaning_embs = st_model.encode(shuffled_meanings, convert_to_numpy=True)
random_scores = [
    1 - cosine(expr_embs[i], random_meaning_embs[i]) for i in range(len(df))
]

df["st_expr_vs_random_meaning"] = random_scores


# Final comparison
print("Correct expression-meaning pairs:", df["st_expr_vs_meaning"].mean())
print("Random expression-meaning pairs:", df["st_expr_vs_random_meaning"].mean())

print("Spearman opacity vs expression-meaning similarity:")
print(spearmanr(df["opacity"], df["st_expr_vs_meaning"]))

plt.figure(figsize=(7, 5))
plt.boxplot(
    [df["st_expr_vs_meaning"], df["st_expr_vs_random_meaning"]],
    labels=["Correct meaning", "Random meaning"]
)
plt.ylabel("Cosine similarity")
plt.title("Expression vs figurative meaning: correct pairs vs random baseline")
plt.grid(True)
plt.show()


The similarity between idiomatic expressions and their correct figurative meanings (mean ≈ 0.50) is higher than the similarity with randomly assigned meanings (mean ≈ 0.42).
This demonstrates that the model captures meaningful semantic relationships between idioms and their figurative interpretations.

The negative correlation between opacity and expression-meaning similarity (Spearman ρ ≈ -0.30, p < 0.001) indicates that more opaque idioms tend to have lower similarity with their figurative meanings.

This is consistent with linguistic expectations: highly opaque idioms are less directly related to their literal or paraphrased meaning, making them harder to represent accurately in embedding space.

# 8. Contextual Analysis

This experiment analyzes how similar the isolated expression is to the context sentence in which it appears.

The question is whether the embedding of the expression alone is close to the embedding of the sentence containing it.

This analysis should be interpreted carefully because a full sentence contains many additional lexical and semantic elements beyond the idiom itself.

Question:
the embedding of isolated expression is different from the embedding of the expression in context?

In [ ]:
# If context_sentence has missing values, replace them with an empty string
df["context_sentence"] = df["context_sentence"].fillna("").astype(str)

df["st_sim_expr_vs_context"] = [
    cosine_sim(encode_text_st(expr), encode_text_st(ctx))
    for expr, ctx in zip(df["expression"], df["context_sentence"])
]

print(df[["expression", "st_sim_expr_vs_context"]].head())
print(df["st_sim_expr_vs_context"].describe())

print("Spearman opacity vs expression-context similarity:")
print(spearmanr(df["opacity"], df["st_sim_expr_vs_context"]))


# Optional categorical comparison
# This divides automatic opacity into three groups: low, medium, high.
df["auto_opacity_mean_cat"] = pd.qcut(
    df["auto_opacity_mean"],
    q=3,
    labels=["low", "medium", "high"],
    duplicates="drop"
)

print(pd.crosstab(df["opacity"], df["auto_opacity_mean_cat"]))

The similarity between the isolated expression and its context sentence is moderate (mean ≈ 0.49), but shows no meaningful correlation with opacity.
This suggests that context similarity is influenced more by sentence-level semantics and lexical overlap than by the idiomatic nature of the expression itself.

# 9. Statistical Analysis

This section summarizes the relationship between:
- manual opacity;
- expression-component similarity;
- automatic opacity;
- expression-meaning similarity.

The goal is to determine whether embedding-based measures align with the manually annotated linguistic opacity scores.


In [ ]:
pearson_res = pearsonr(df["opacity"], df["st_expr_vs_meaning"])
spearman_res = spearmanr(df["opacity"], df["st_expr_vs_meaning"])

print("Manual opacity vs expression-meaning similarity")
print("Pearson r =", pearson_res.statistic, "p =", pearson_res.pvalue)
print("Spearman rho =", spearman_res.statistic, "p =", spearman_res.pvalue)

print("Spearman opacity vs expression-component similarity:")
print(spearmanr(df["opacity"], df["st_sim_expr_vs_mean"]))

print("Spearman opacity vs automatic opacity:")
print(spearmanr(df["opacity"], df["auto_opacity_mean"]))

# Kruskal-Wallis test across opacity groups
# This tests whether expression-meaning similarity differs across opacity levels.
groups = [group["st_expr_vs_meaning"].values for _, group in df.groupby("opacity")]
kruskal_res = kruskal(*groups)

print("Kruskal-Wallis H =", kruskal_res.statistic, "p =", kruskal_res.pvalue)


The categorical comparison between manual opacity and automatically derived opacity shows substantial overlap between categories.
This indicates that the embedding-based measure cannot reliably distinguish between low, medium, and high opacity idioms.

The significant negative correlation between opacity and expression-meaning similarity confirms that more opaque idioms are less aligned with their figurative meanings in embedding space.
In contrast, no significant relationship was found between opacity and expression-component similarity.

The Kruskal-Wallis test (p < 0.001) indicates that similarity scores differ significantly across opacity groups, confirming that opacity has a measurable effect on embedding behavior.

# 10. Similarity Matrices

Similarity matrices are used for qualitative exploration.
They show how expressions relate to each other, and how expressions relate to their figurative meanings.

Only a small random subset is visualized to keep the heatmaps readable.

In [ ]:
import seaborn as sns

subset_df = df.sample(min(30, len(df)), random_state=42).reset_index(drop=True)

subset_expr_embs = [encode_text_st(expr) for expr in subset_df["expression"]]
sim_matrix_expr = cosine_similarity(subset_expr_embs)

plt.figure(figsize=(12, 10))
sns.heatmap(
    sim_matrix_expr,
    xticklabels=subset_df["expression"],
    yticklabels=subset_df["expression"]
)
plt.title("Similarity matrix: expressions vs expressions")
plt.xticks(rotation=90)
plt.yticks(rotation=0)
plt.show()

Expressions vs figurative meaning


In [ ]:
subset_meaning_embs = [encode_text_st(m) for m in subset_df["figurative_meaning"]]
sim_matrix_expr_meaning = cosine_similarity(subset_expr_embs, subset_meaning_embs)

plt.figure(figsize=(12, 10))
sns.heatmap(
    sim_matrix_expr_meaning,
    xticklabels=subset_df["figurative_meaning"],
    yticklabels=subset_df["expression"]
)
plt.title("Similarity matrix: expressions vs figurative meanings")
plt.xticks(rotation=90)
plt.yticks(rotation=0)
plt.show()

Expression vs mean components

In [ ]:
subset_mean_embs = []
for expr in subset_df["expression"]:
    words = tokenize_expression(expr)
    word_embs = encode_words_st(words)
    subset_mean_embs.append(np.mean(word_embs, axis=0))

sim_matrix_expr_components = cosine_similarity(subset_expr_embs, subset_mean_embs)

plt.figure(figsize=(12, 10))
sns.heatmap(
    sim_matrix_expr_components,
    xticklabels=subset_df["expression"],
    yticklabels=subset_df["expression"]
)
plt.title("Similarity matrix: expressions vs mean component embeddings")
plt.xticks(rotation=90)
plt.yticks(rotation=0)
plt.show()

# 11. Comparison Across Models

This section compares SentenceTransformer with two transformer encoders:
- xlm-roberta-base;
- dumitrescustefan/bert-base-romanian-cased-v1.

Important methodological note:
These models are not sentence similarity models by default. They are contextual transformer encoders. To obtain one vector for a whole text, we apply mean pooling over the last hidden states.

Therefore, their results should be interpreted as exploratory rather than as direct replacements for SentenceTransformer.

XLM Roberta base model

In [ ]:
!pip install -q transformers torch scikit-learn

import torch
from transformers import AutoTokenizer, AutoModel

def get_transformer_embedding(text, tokenizer, model):
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=64
    )

    with torch.no_grad():
        outputs = model(**inputs)

    last_hidden = outputs.last_hidden_state
    attention_mask = inputs["attention_mask"].unsqueeze(-1)

    masked_hidden = last_hidden * attention_mask
    summed = masked_hidden.sum(dim=1)
    counts = attention_mask.sum(dim=1)
    mean_pooled = summed / counts

    return mean_pooled[0].numpy()


In [ ]:
# -----------------------------
# XLM-RoBERTa base
# -----------------------------

xlmr_name = "xlm-roberta-base"
xlmr_tokenizer = AutoTokenizer.from_pretrained(xlmr_name)
xlmr_model = AutoModel.from_pretrained(xlmr_name)
xlmr_model.eval()

xlmr_similarity = []

for _, row in df.iterrows():
    expr_emb = get_transformer_embedding(row["expression"], xlmr_tokenizer, xlmr_model)
    meaning_emb = get_transformer_embedding(row["figurative_meaning"], xlmr_tokenizer, xlmr_model)
    score = 1 - cosine(expr_emb, meaning_emb)
    xlmr_similarity.append(score)


df["xlmr_expr_vs_meaning"] = xlmr_similarity

print(df[["expression", "opacity", "xlmr_expr_vs_meaning"]].head(10))
print()
print(df.groupby("opacity")["xlmr_expr_vs_meaning"].mean())
print()
print("Pearson:", df["opacity"].corr(df["xlmr_expr_vs_meaning"], method="pearson"))
print("Spearman:", df["opacity"].corr(df["xlmr_expr_vs_meaning"], method="spearman"))


The XLM-RoBERTa model produces extremely high similarity scores (≈ 0.99) for all expression-meaning pairs, regardless of opacity level.
The near-constant values indicate that the model fails to discriminate between different semantic relationships in this task.

This suggests that XLM-RoBERTa, when used with simple mean pooling, is not suitable for capturing idiomatic meaning or semantic opacity.

In [ ]:
# -----------------------------
# Romanian BERT
# -----------------------------

ro_model_name = "dumitrescustefan/bert-base-romanian-cased-v1"
ro_tokenizer = AutoTokenizer.from_pretrained(ro_model_name)
ro_model = AutoModel.from_pretrained(ro_model_name)
ro_model.eval()

ro_similarity = []

for _, row in df.iterrows():
    expr_emb = get_transformer_embedding(row["expression"], ro_tokenizer, ro_model)
    meaning_emb = get_transformer_embedding(row["figurative_meaning"], ro_tokenizer, ro_model)
    score = 1 - cosine(expr_emb, meaning_emb)
    ro_similarity.append(score)

df["robert_ro_expr_vs_meaning"] = ro_similarity

print(df[["expression", "opacity", "robert_ro_expr_vs_meaning"]].head(10))
print()
print(df.groupby("opacity")["robert_ro_expr_vs_meaning"].mean())
print()
print("Pearson:", df["opacity"].corr(df["robert_ro_expr_vs_meaning"], method="pearson"))
print("Spearman:", df["opacity"].corr(df["robert_ro_expr_vs_meaning"], method="spearman"))


The Romanian BERT-based model shows a moderate negative correlation between idiomatic opacity and expression-meaning similarity (Pearson r ≈ -0.27, Spearman ρ ≈ -0.27).
This indicates that as the opacity of an idiomatic expression increases, its semantic similarity to the annotated figurative meaning decreases.

# 12. Results + interpretation

The experiments indicate that Romanian idiomatic expressions can be analyzed through the relationship between the embedding of the full expression, the embeddings of its component words, and the embedding of its figurative meaning.

The compositionality experiment uses the similarity between the full expression and the average of its component-word embeddings as an approximation of semantic transparency. When this similarity is lower, the expression may be interpreted as more semantically opaque in the embedding space.

Automatic opacity, computed as 1 minus this similarity, provides a quantitative measure that can be compared with manually annotated opacity scores. However, this measure should not be treated as equivalent to human linguistic judgment. It is an embedding-based approximation and may be influenced by lexical overlap, word frequency, model training data, and tokenization.

The expression-versus-meaning experiment evaluates whether the model places an idiomatic expression close to its figurative paraphrase. The random baseline is important because it shows whether correct expression-meaning pairs are more semantically aligned than unrelated pairs.

The comparison across models should be interpreted carefully. SentenceTransformer is designed for semantic similarity, while XLM-RoBERTa and Romanian BERT are general transformer encoders. Their sentence-level embeddings are obtained through mean pooling, which is a practical but imperfect method.

Overall, the results can support a discussion about both the potential and the limitations of distributed models in representing Romanian figurative language. The corpus itself is also a useful contribution, since Romanian idiomatic expressions remain underrepresented in computational linguistic resources.


In [ ]:
output_file = "romanian_idioms_results_corrected.xlsx"
df.to_excel(output_file, index=False)
print(f"Results were saved to {output_file}")